# Compute Haversine distances, Country and State accuracies for original images and filtered images

In [ ]:
import haversine
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from glob import glob
import os
import reverse_geocoder as rg

## Setup

In [ ]:
os.makedirs("./results", exist_ok=True)

In [ ]:
def calculate_haversine(coord1, coord2):
    """Calculate the Haversine distance between two (lat, lon) coordinates."""
    return haversine.haversine(coord1, coord2, unit=haversine.Unit.KILOMETERS)

In [ ]:
def get_latitude_longitude_from_image_id(image_id, original=False):
    """Extract latitude and longitude from image ID."""
    if original:
        _id, lat_str, lon_str = image_id.split("_")
    else:
        _id, lat_str, lon_str, method = image_id.split("_")
    return float(lat_str), float(lon_str)

In [ ]:
def get_latitude_longitude_from_computed_output(computed_output):
    """Extract latitude and longitude from computed output string."""
    computed_output_tuple = eval(computed_output)
    lat_str, lon_str = computed_output_tuple[0], computed_output_tuple[1]
    return float(lat_str), float(lon_str)

## Calculate Haversine distances

In [ ]:
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(f"./csv_files/osmclip_{method}.csv")
    haversine_distances = []
    flag = False
    if method == "original":
        flag = True
    for _, row in df.iterrows():
        image_id = row["image_id"]
        computed_output = row["comp_coords"]

        true_latlon = get_latitude_longitude_from_image_id(image_id, original=flag)
        pred_latlon = get_latitude_longitude_from_computed_output(computed_output)

        distance = calculate_haversine(true_latlon, pred_latlon)
        haversine_distances.append(distance)

    df["haversine_distance_km"] = haversine_distances
    df.to_csv(os.path.join("./results", f"osmclip_haversine_{method}.csv"), index=False)

In [ ]:
# Print average haversine distances for each method
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(os.path.join("./results", f"osmclip_haversine_{method}.csv"))
    avg_distance = df["haversine_distance_km"].mean()
    print(f"Average Haversine distance for {method}: {avg_distance:.2f} km")

In [ ]:
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(f"./csv_files/geoclip_{method}.csv")
    haversine_distances = []
    flag = False
    if method == "original":
        flag = True
    for _, row in df.iterrows():
        image_id = row["image_id"]
        computed_output = row["comp_coords"]

        true_latlon = get_latitude_longitude_from_image_id(image_id, original=flag)
        pred_latlon = get_latitude_longitude_from_computed_output(computed_output)

        distance = calculate_haversine(true_latlon, pred_latlon)
        haversine_distances.append(distance)

    df["haversine_distance_km"] = haversine_distances
    df.to_csv(os.path.join("./results", f"geoclip_haversine_{method}.csv"), index=False)

In [ ]:
# Print average haversine distances for each method
for method in ["original", "clarendon", "willow", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(os.path.join("./results", f"geoclip_haversine_{method}.csv"))
    avg_distance = df["haversine_distance_km"].mean()
    print(f"Average Haversine distance for {method}: {avg_distance:.2f} km")

In [ ]:
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(f"osmclip_{method}.csv")
    all_ids = []
    all_gt_coordinates = []
    all_pred_coordinates = []
    flag = False
    if method == "original":
        flag = True
    for _, row in df.iterrows():
        image_id = row["image_id"]
        computed_output = row["comp_coords"]
        raw_id = image_id.split("_")[0]
        all_ids.append(raw_id)

        true_latlon = get_latitude_longitude_from_image_id(image_id, original=flag)
        pred_latlon = get_latitude_longitude_from_computed_output(computed_output)

        all_gt_coordinates.append(true_latlon)
        all_pred_coordinates.append(pred_latlon)

    # Create a new DataFrame with the collected data
    result_df = pd.DataFrame(
        {
            "id": all_ids,
            "gt_coordinates": all_gt_coordinates,
            "pred_coordinates": all_pred_coordinates,
        }
    )

    # Query reverse geocoder for location names
    gt_coords_list = [coord for coord in all_gt_coordinates]
    gt_locations = rg.search(gt_coords_list)
    gt_country_codes = [loc["cc"] for loc in gt_locations]
    gt_admin1_codes = [loc["admin1"] for loc in gt_locations]

    pred_coords_list = [coord for coord in all_pred_coordinates]
    pred_locations = rg.search(pred_coords_list)
    pred_country_codes = [loc["cc"] for loc in pred_locations]
    pred_admin1_codes = [loc["admin1"] for loc in pred_locations]

    result_df["gt_country_code"] = gt_country_codes
    result_df["pred_country_code"] = pred_country_codes
    result_df["gt_admin1_code"] = gt_admin1_codes
    result_df["pred_admin1_code"] = pred_admin1_codes

    # Save results to CSV
    result_df.to_csv(
        os.path.join("./results", f"osmclip_locations_{method}.csv"), index=False
    )


In [ ]:
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(f"geoclip_{method}.csv")
    all_ids = []
    all_gt_coordinates = []
    all_pred_coordinates = []
    flag = False
    if method == "original":
        flag = True
    for _, row in df.iterrows():
        image_id = row["image_id"]
        computed_output = row["comp_coords"]
        raw_id = image_id.split("_")[0]
        all_ids.append(raw_id)

        true_latlon = get_latitude_longitude_from_image_id(image_id, original=flag)
        pred_latlon = get_latitude_longitude_from_computed_output(computed_output)

        all_gt_coordinates.append(true_latlon)
        all_pred_coordinates.append(pred_latlon)

    # Create a new DataFrame with the collected data
    result_df = pd.DataFrame(
        {
            "id": all_ids,
            "gt_coordinates": all_gt_coordinates,
            "pred_coordinates": all_pred_coordinates,
        }
    )

    # Query reverse geocoder for location names
    gt_coords_list = [coord for coord in all_gt_coordinates]
    gt_locations = rg.search(gt_coords_list)
    gt_country_codes = [loc["cc"] for loc in gt_locations]
    gt_admin1_codes = [loc["admin1"] for loc in gt_locations]

    pred_coords_list = [coord for coord in all_pred_coordinates]
    pred_locations = rg.search(pred_coords_list)
    pred_country_codes = [loc["cc"] for loc in pred_locations]
    pred_admin1_codes = [loc["admin1"] for loc in pred_locations]

    result_df["gt_country_code"] = gt_country_codes
    result_df["pred_country_code"] = pred_country_codes
    result_df["gt_admin1_code"] = gt_admin1_codes
    result_df["pred_admin1_code"] = pred_admin1_codes

    # Save results to CSV
    result_df.to_csv(
        os.path.join("./results", f"geoclip_locations_{method}.csv"), index=False
    )


## Compute Country and State accuracies

In [ ]:
# Compute accuracies at country and admin1 levels for each method and save to a summary CSV
summary_data = []
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(os.path.join("./results", f"osmclip_locations_{method}.csv"))
    country_accuracy = (df["gt_country_code"] == df["pred_country_code"]).mean()
    admin1_accuracy = (df["gt_admin1_code"] == df["pred_admin1_code"]).mean()
    summary_data.append(
        {
            "method": method,
            "country_accuracy": country_accuracy,
            "admin1_accuracy": admin1_accuracy,
        }
    )

summary_df_1 = pd.DataFrame(summary_data)
summary_df_1.to_csv("./results/osmclip_summary.csv", index=False)


In [ ]:
summary_df_1

In [ ]:
# Compute accuracies at country and admin1 levels for each method and save to a summary CSV
summary_data = []
for method in ["original", "willow", "clarendon", "gingham", "inkwell", "lark", "moon"]:
    df = pd.read_csv(os.path.join("./results", f"geoclip_locations_{method}.csv"))
    country_accuracy = (df["gt_country_code"] == df["pred_country_code"]).mean()
    admin1_accuracy = (df["gt_admin1_code"] == df["pred_admin1_code"]).mean()
    summary_data.append(
        {
            "method": method,
            "country_accuracy": country_accuracy,
            "admin1_accuracy": admin1_accuracy,
        }
    )

summary_df_2 = pd.DataFrame(summary_data)
summary_df_2.to_csv("./results/geoclip_summary.csv", index=False)

In [ ]:
summary_df_2